# 🧔 Employee Risk Prediction using Support Vector Machines (SVM)

## A practical, real‑world inspired HR + Operations analytics notebook

Let’s be clear from the start 🙂  
This notebook is **not** about judging individuals or automating HR decisions.  
It is about **understanding**, **visualizing**, and **reasoning** about how employee risk can emerge when **behavioral, operational, and structural factors** drift together.

**No black magic.**  
Just patterns across people, processes, and context.

## 1. Problem framing (why this is not linear)

Employee “risk” (disengagement, low performance, or potential attrition) is rarely caused by a single factor.  
In practice, it tends to appear in **specific combinations**, for example:

- Low engagement **and** limited training  
- High scrap exposure **and** unstable staffing conditions  
- Night shift **and** high area rotation  
- Temporary/outsourced contracts **and** weak performance signals  

A simple rule like “if punctuality < 0.90 then risk” is usually **wrong**, because real systems are multivariate.

That is exactly where **SVM** becomes useful: it can learn **non‑linear decision regions** (risk “zones”) where combinations of factors align.

## 2. Why SVM here?

SVM helps when:
- Risk patterns are **non‑linear**
- The boundary between “risk” and “no risk” depends on **interactions**
- You care about **decision regions** rather than a single threshold

Think of it this way:  
We are not asking “which single variable causes risk?”  
We are asking “**which combinations of conditions** create risk‑prone zones?”

## 3. Setup

We’ll use:
- pandas / numpy for data
- scikit‑learn for preprocessing + SVM + tuning
- matplotlib / seaborn for plots
- SHAP for interpretability (optional if not installed)
- plotly for interactive 3D visualization (optional)

In [ ]:
# Core
import numpy as np
import pandas as pd

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Modeling
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC, LinearSVC
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)

sns.set(style="whitegrid")
plt.rcParams["figure.figsize"] = (8, 5)
pd.set_option("display.max_columns", 200)

print("Libraries loaded.")

In [ ]:
# Load dataset (CSV)

DATA_PATH = "hr_risk_svm_data.csv"  # <-- change if needed
df = pd.read_csv(DATA_PATH)

df.head()

## 4. Quick dataset check

We validate:
- shape and data types
- class balance (risk rate)
- summary statistics

In [ ]:
df.shape, df.dtypes

In [ ]:
df["high_risk"].value_counts().rename("count")

In [ ]:
df["high_risk"].value_counts(normalize=True).rename("proportion")

In [ ]:
df.describe(include="all").T.head(20)

## 5. Exploratory Data Analysis (EDA) with intent

We are not plotting “just because”.  
We want to answer:

- Which variables separate risk vs no risk clearly?
- Which variables only matter in combination?
- Are there structural patterns that dominate personal factors?

In [ ]:
numeric_cols = [
    "training_hours_annual","punctuality_ratio","productivity_index",
    "scrap_associated_pct","engagement_score","years_experience","area_rotation_rate"
]

for col in numeric_cols:
    plt.figure()
    sns.histplot(data=df, x=col, hue="high_risk", kde=True, stat="density", common_norm=False)
    plt.title(f"Distribution of {col} by high_risk")
    plt.tight_layout()
    plt.show()

In [ ]:
for col in ["area","shift","contract_type"]:
    plt.figure()
    order = df[col].value_counts().index
    sns.countplot(data=df, x=col, hue="high_risk", order=order)
    plt.title(f"{col} distribution by high_risk")
    plt.xticks(rotation=15)
    plt.tight_layout()
    plt.show()

## 6. Train/test split and preprocessing

SVM is distance‑ and geometry‑sensitive, so:
- Numeric features must be **scaled**
- Categorical features must be **encoded** (one‑hot)

We use a scikit‑learn Pipeline to avoid leakage and keep everything reproducible.

In [ ]:
X = df.drop(columns=["high_risk"])
y = df["high_risk"]

num_features = [
    "training_hours_annual","punctuality_ratio","productivity_index",
    "scrap_associated_pct","engagement_score","years_experience","area_rotation_rate"
]
cat_features = ["area","shift","contract_type"]

preprocess = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), num_features),
        ("cat", OneHotEncoder(drop="first"), cat_features)
    ]
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

X_train.shape, X_test.shape

## 7. Baseline SVM (RBF) before tuning

We start with a reasonable baseline.  
Then we tune **C** and **gamma** (and compare linear vs RBF).

In [ ]:
svm_base = Pipeline(steps=[
    ("pre", preprocess),
    ("clf", SVC(kernel="rbf", C=5, gamma="scale", probability=True, random_state=42))
])

svm_base.fit(X_train, y_train)

pred_test = svm_base.predict(X_test)
print("Baseline SVM (RBF)")
print("Accuracy:", accuracy_score(y_test, pred_test))
print("F1      :", f1_score(y_test, pred_test))

## 8. Hyperparameter tuning (GridSearchCV)

We tune:
- kernel ∈ {linear, rbf}
- C controls margin hardness
- gamma controls how “local” the RBF influence is

We optimize for **F1** (balanced view of precision/recall).

In [ ]:
param_grid = {
    "clf__kernel": ["linear", "rbf"],
    "clf__C": [0.1, 1, 10, 50],
    "clf__gamma": ["scale", 0.1, 0.01]  # ignored for linear
}

svm_pipe = Pipeline(steps=[
    ("pre", preprocess),
    ("clf", SVC(probability=True, random_state=42))
])

grid = GridSearchCV(
    estimator=svm_pipe,
    param_grid=param_grid,
    scoring="f1",
    cv=5,
    n_jobs=-1
)

grid.fit(X_train, y_train)
grid.best_params_, grid.best_score_

In [ ]:
best_model = grid.best_estimator_
pred_test = best_model.predict(X_test)
pred_train = best_model.predict(X_train)

print("Best SVM (after tuning)")
print("Train Accuracy:", accuracy_score(y_train, pred_train))
print("Test  Accuracy:", accuracy_score(y_test, pred_test))
print("Test  Precision:", precision_score(y_test, pred_test))
print("Test  Recall   :", recall_score(y_test, pred_test))
print("Test  F1       :", f1_score(y_test, pred_test))
print("\nClassification report (test):\n")
print(classification_report(y_test, pred_test))

In [ ]:
cm = confusion_matrix(y_test, pred_test)
plt.figure()
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=[0,1], yticklabels=[0,1])
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix — high_risk")
plt.tight_layout()
plt.show()

## 9. Interpretability: Linear SVM coefficients (directional insight)

SVM with an RBF kernel is powerful but less directly interpretable.  
A common strategy is to train a **linear SVM** (same preprocessing) to understand which directions push predictions toward risk.

This is not “the truth”, but it is a useful **directional diagnostic**.

In [ ]:
svm_linear = Pipeline(steps=[
    ("pre", preprocess),
    ("clf", LinearSVC(C=1.0, random_state=42))
])

svm_linear.fit(X_train, y_train)

ohe = svm_linear.named_steps["pre"].named_transformers_["cat"]
cat_names = ohe.get_feature_names_out(cat_features)
feature_names = num_features + list(cat_names)

coefs = svm_linear.named_steps["clf"].coef_.ravel()
importance = pd.DataFrame({"feature": feature_names, "coef": coefs}).sort_values("coef", ascending=False)
importance.head(15)

In [ ]:
plt.figure(figsize=(10,6))
sns.barplot(data=importance.head(20), x="coef", y="feature")
plt.axvline(0, color="black", linewidth=1)
plt.title("Top positive coefficients (push toward high_risk) — Linear SVM")
plt.tight_layout()
plt.show()

plt.figure(figsize=(10,6))
sns.barplot(data=importance.tail(20), x="coef", y="feature")
plt.axvline(0, color="black", linewidth=1)
plt.title("Top negative coefficients (push toward no risk) — Linear SVM")
plt.tight_layout()
plt.show()

## 10 SHAP: model explanations (optional but powerful)

**SHAP** explains predictions as additive contributions (like forces) that push the model output toward risk or no risk.

If SHAP is not installed, skip this section.  
(You can install with: `pip install shap`)

In [ ]:
# Optional: SHAP
try:
    import shap
    shap_installed = True
    print("SHAP is installed.")
except Exception as e:
    shap_installed = False
    print("SHAP not installed. Skipping SHAP section. Error:", e)

In [ ]:
# Optional: SHAP
try:
    import shap
    shap_installed = True
    print("SHAP is installed.")
except Exception as e:
    shap_installed = False
    print("SHAP not installed. Skipping SHAP section. Error:", e)

if shap_installed:
    # --- Robust sampling (works even if dataset is small) ---
    n_train = min(200, len(X_train))
    n_test  = min(200, len(X_test))

    X_train_sample = X_train.sample(n=n_train, random_state=42)
    X_test_sample  = X_test.sample(n=n_test, random_state=42)

    # Probability of class 1 (high_risk)
    f_model = lambda data: best_model.predict_proba(
        pd.DataFrame(data, columns=X_train.columns)
    )[:, 1]

    explainer = shap.KernelExplainer(model=f_model, data=X_train_sample)

    # For small datasets you can reduce nsamples to speed up
    nsamples = 100 if n_test >= 100 else max(50, n_test)

    shap_values = explainer.shap_values(X_test_sample, nsamples=nsamples)

    shap.summary_plot(shap_values, X_test_sample, plot_type="dot", max_display=15)

## 11. Decision regions (2D) — when visualization is honest

A 2D decision boundary is **not the full model**, but it can be educational.

We choose pairs that are meaningful:
- engagement_score vs scrap_associated_pct  
- punctuality_ratio vs engagement_score

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler

def plot_svm_boundary_2d(df_all, x_col, y_col, y_target="high_risk", C=5, gamma="scale"):
    X2 = df_all[[x_col, y_col]]
    y2 = df_all[y_target]

    X_train2, X_test2, y_train2, y_test2 = train_test_split(
        X2, y2, test_size=0.25, random_state=42, stratify=y2
    )

    model2 = Pipeline(steps=[
        ("scaler", StandardScaler()),
        ("clf", SVC(kernel="rbf", C=C, gamma=gamma, probability=True, random_state=42))
    ])
    model2.fit(X_train2, y_train2)

    x_min, x_max = X2[x_col].min(), X2[x_col].max()
    y_min, y_max = X2[y_col].min(), X2[y_col].max()

    pad_x = (x_max - x_min) * 0.05 if x_max > x_min else 0.05
    pad_y = (y_max - y_min) * 0.05 if y_max > y_min else 0.05

    xx, yy = np.meshgrid(
        np.linspace(x_min - pad_x, x_max + pad_x, 300),
        np.linspace(y_min - pad_y, y_max + pad_y, 300)
    )

    grid = pd.DataFrame({x_col: xx.ravel(), y_col: yy.ravel()})
    Z = model2.predict(grid).reshape(xx.shape)

    plt.figure(figsize=(7,5))
    plt.contourf(xx, yy, Z, levels=[-0.5, 0.5, 1.5], alpha=0.25, cmap="bwr")
    sns.scatterplot(
        data=X_test2.assign(high_risk=y_test2.values),
        x=x_col, y=y_col, hue="high_risk", palette={0:"tab:blue", 1:"tab:red"},
        alpha=0.75
    )
    plt.title(f"SVM decision regions ({x_col} vs {y_col})")
    plt.tight_layout()
    plt.show()

plot_svm_boundary_2d(df, "engagement_score", "scrap_associated_pct")
plot_svm_boundary_2d(df, "punctuality_ratio", "engagement_score")

## 12. Scenario simulation (“what if?”)

This is where the notebook becomes a **decision‑support conversation tool**:

- “What if training hours increase?”  
- “What if we move a person to night shift?”  
- “What if engagement drops but contract becomes permanent?”

We predict:
- class (risk / no risk)
- probability of risk

In [ ]:
def simulate_employee(profile: dict, model=best_model):
    # profile must contain all feature keys used in training
    X_one = pd.DataFrame([profile])
    pred_class = int(model.predict(X_one)[0])
    pred_prob = float(model.predict_proba(X_one)[0][1])  # probability of high_risk
    return pred_class, pred_prob

example_low_risk = {
    "training_hours_annual": 50,
    "punctuality_ratio": 0.99,
    "productivity_index": 110,
    "scrap_associated_pct": 2,
    "engagement_score": 5,
    "years_experience": 10,
    "area_rotation_rate": 0.05,
    "area": "Quality",
    "shift": "Morning",
    "contract_type": "Permanent"
}

example_high_risk = {
    "training_hours_annual": 5,
    "punctuality_ratio": 0.78,
    "productivity_index": 80,
    "scrap_associated_pct": 10,
    "engagement_score": 1,
    "years_experience": 1,
    "area_rotation_rate": 0.30,
    "area": "Production",
    "shift": "Night",
    "contract_type": "Outsourcing"
}

print("Example A (expected low risk):", simulate_employee(example_low_risk))
print("Example B (expected high risk):", simulate_employee(example_high_risk))

## Technical Takeaways (Short)

### What this model **is**
- A way to detect **risk patterns** across personal, operational, and structural drivers
- A diagnostic tool that helps ask better questions
- A teaching case for non‑linear classification + interpretability

### What this model **is not**
- A replacement for HR decision processes
- A “truth engine” about individuals
- A punitive automation tool

In real HR analytics, the right use case is:
- **support interventions** (training, team support, workload redesign)
- **identify systemic risk zones** (areas with high rotation, unstable contracts)
- **prioritize root cause analysis**, not punish people

—
LozanoLsa
Regards from MX